# EXAMPLE KIRCHHOF DIAGRAMM

[back](chapter3.3.md)

In [1]:
import pypsa

# 1. Αρχικοποίηση Δικτύου και Χρόνου
network = pypsa.Network()
network.set_snapshots([0])  # Μόνο ένα χρονικό βήμα (η στιγμή 't' του διαγράμματος)

# 2. Προσθήκη Κόμβων (Buses)
network.add("Bus", "Bus_A", v_nom=380)
network.add("Bus", "Bus_B", v_nom=380)
network.add("Bus", "Bus_C", v_nom=380)

# 3. Προσθήκη Εξοπλισμού στον Κόμβο Α
# Γεννήτρια g_A (παράγει ρεύμα)
network.add("Generator", "g_A", bus="Bus_A", p_nom=1000, marginal_cost=50)
# Φορτίο d_A (καταναλώνει ρεύμα)
network.add("Load", "d_A", bus="Bus_A", p_set=200)

# 4. Προσθήκη Εξοπλισμού στον Κόμβο C
# Φορτίο d_C (καταναλώνει ρεύμα) - Ας πούμε ότι η πόλη C χρειάζεται 400 MW
network.add("Load", "d_C", bus="Bus_C", p_set=400)
# Το στοιχείο "C" (π.χ. μια μικρή ακριβή εφεδρική γεννήτρια)
network.add("Generator", "Component_C", bus="Bus_C", p_nom=500, marginal_cost=100)

# 5. Προσθήκη Γραμμών Μεταφοράς (Lines) - Εδώ μπαίνει η Αντίδραση (x)
# Γραμμή Α-Β: Ορίζουμε την αντίδραση (x) και τη μέγιστη χωρητικότητα (s_nom)
network.add(
    "Line", "Line_AB",
    bus0="Bus_A", bus1="Bus_B",
    x=0.1,         # Η αντίδραση (reactance) x_AB
    s_nom=1000     # Μέγιστη χωρητικότητα σε MVA
)

# Γραμμή Β-C
network.add(
    "Line", "Line_BC",
    bus0="Bus_B", bus1="Bus_C",
    x=0.15,        # Ας πούμε ότι αυτή η γραμμή έχει λίγο μεγαλύτερη αντίδραση
    s_nom=1000
)

# 6. Επίλυση του Συστήματος (Γραμμική Βελτιστοποίηση - LOPF)
print("Υπολογισμός Ροών (LOPF)...\n")
network.optimize()

# 7. Εκτύπωση των Αποτελεσμάτων
print("--- ΡΟΕΣ ΙΣΧΥΟΣ ΣΤΙΣ ΓΡΑΜΜΕΣ (f_AB, f_BC) ---")
# Το p0 είναι η ενεργός ισχύς που μπαίνει στη γραμμή (power flow)
print(network.lines_t.p0)

print("\n--- ΓΩΝΙΕΣ ΤΑΣΗΣ ΣΤΟΥΣ ΚΟΜΒΟΥΣ (θ) ---")
# Το v_ang είναι η γωνία φάσης της τάσης (voltage angle) σε ακτίνια
print(network.buses_t.v_ang)

Index(['Bus_A', 'Bus_B', 'Bus_C'], dtype='str', name='name')
Index(['Line_AB', 'Line_BC'], dtype='str', name='name')
Index(['Line_AB', 'Line_BC'], dtype='str', name='name')


Υπολογισμός Ροών (LOPF)...



INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 11 duals
Objective: 3.00e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


--- ΡΟΕΣ ΙΣΧΥΟΣ ΣΤΙΣ ΓΡΑΜΜΕΣ (f_AB, f_BC) ---
name      Line_AB  Line_BC
snapshot                  
0           400.0    400.0

--- ΓΩΝΙΕΣ ΤΑΣΗΣ ΣΤΟΥΣ ΚΟΜΒΟΥΣ (θ) ---
name         Bus_A     Bus_B     Bus_C
snapshot                              
0         0.000323  0.000046 -0.000369


1. Τα Μηνύματα του Επιλύτη (Solver) Highs

    Status: ok & Termination condition: optimal: Tο πρόγραμμα βρήκε την απόλυτα βέλτιστη και φθηνότερη λύση, χωρίς να παραβιάσει κανέναν κανόνα (π.χ. δεν υπερφόρτωσε κανένα καλώδιο).

    Objective: 3.00e+04: Αυτό είναι το συνολικό κόστος λειτουργίας του συστήματος. Γράφεται με επιστημονική μορφή και σημαίνει 30.000 €.

        Γιατί 30.000; Η συνολική ζήτηση ήταν 600 MW (200 MW στον Κόμβο Α + 400 MW στον Κόμβο C). Το σύστημα επέλεξε να χρησιμοποιήσει μόνο τη φθηνή γεννήτρια gA​ (που κοστίζει 50 €/MWh). Άρα: 600 MW × 50 €/MWh = 30.000 €. Η ακριβή γεννήτρια στον Κόμβο C (100 €/MWh) έμεινε σβηστή!
    
2. Οι Ροές Ισχύος (Power Flows)

Εδώ βλέπουμε πόσο ρεύμα πέρασε μέσα από τα καλώδια.

    Line_AB: 400.0 (Μεταφορά από τον Κόμβο Α στον Κόμβο Β)

    Line_BC: 400.0 (Μεταφορά από τον Κόμβο Β στον Κόμβο C)

Το αποτέλεσμα είναι απόλυτα λογικό: Ο Κόμβος Α παρήγαγε συνολικά 600 MW. Κράτησε τα 200 MW για τον εαυτό του (τη δική του ζήτηση dA​) και έστειλε τα υπόλοιπα 400 MW να ταξιδέψουν μέσα από τη Γραμμή Α-Β και μετά από τη Γραμμή Β-C, ώστε να φτάσουν στον Κόμβο C που τα είχε ανάγκη.

3. Οι Γωνίες Τάσης (θ)

Εδώ βλέπουμε τους αριθμούς που επιβεβαιώνουν τον Νόμο του Kirchhoff που συζητήσαμε πριν:

    Bus_A: 0.000323

    Bus_B: 0.000046

    Bus_C: -0.000369

Το PyPSA δημιούργησε μια μαθηματική "κατηφόρα"! Η γωνία στον Κόμβο Α (θA​) είναι η μεγαλύτερη, του Κόμβου Β (θB​) είναι μεσαία, και του Κόμβου C (θC​) είναι η μικρότερη (μάλιστα αρνητική).

Επειδή θA​>θB​>θC​, η ενέργεια αναγκάζεται (βάσει των νόμων της φυσικής) να ταξιδέψει προς αυτή ακριβώς την κατεύθυνση, λύνοντας το πρόβλημά μας τέλεια.

--- 

# STRESS TEST

Στο προηγούμενο σενάριο, η Γραμμή Α-Β μετέφερε άνετα 400 MW. Τώρα, θα υποθέσουμε ότι είναι ένα παλιότερο ή λεπτότερο καλώδιο που αντέχει το πολύ 300 MW (s_nom = 300).

Ο Κόμβος C χρειάζεται ακόμα 400 MW. Εφόσον δεν μπορούν να περάσουν όλα από τη γραμμή, το PyPSA θα αναγκαστεί να βρει άλλη λύση για να μη βυθιστεί η πόλη C στο σκοτάδι (blackout).

In [2]:
import pypsa

network = pypsa.Network()
network.set_snapshots([0])

# Κόμβοι
network.add("Bus", "Bus_A", v_nom=380)
network.add("Bus", "Bus_B", v_nom=380)
network.add("Bus", "Bus_C", v_nom=380)

# Εξοπλισμός Κόμβου Α (Φθηνή παραγωγή 50 €/MWh)
network.add("Generator", "g_A", bus="Bus_A", p_nom=1000, marginal_cost=50)
network.add("Load", "d_A", bus="Bus_A", p_set=200)

# Εξοπλισμός Κόμβου C (Ακριβή παραγωγή 100 €/MWh)
network.add("Load", "d_C", bus="Bus_C", p_set=400)
network.add("Generator", "Component_C", bus="Bus_C", p_nom=500, marginal_cost=100)

# Γραμμές - ΕΔΩ ΕΙΝΑΙ Η ΑΛΛΑΓΗ! Η Line_AB αντέχει πλέον μόνο 300 MW
network.add("Line", "Line_AB", bus0="Bus_A", bus1="Bus_B", x=0.1, s_nom=300)
network.add("Line", "Line_BC", bus0="Bus_B", bus1="Bus_C", x=0.15, s_nom=1000)

# Επίλυση
network.optimize()

# Αποτελέσματα
print("\n--- ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ (MW) ---")
print(network.generators_t.p)

print("\n--- ΡΟΕΣ ΙΣΧΥΟΣ ΣΤΙΣ ΓΡΑΜΜΕΣ (MW) ---")
print(network.lines_t.p0)


print("\n--- ΟΡΙΑΚΕΣ ΤΙΜΕΣ ΚΟΜΒΩΝ (LMP σε €/MWh) ---")
# Η εντολή marginal_price τραβάει τις "σκιώδεις τιμές" (shadow prices) του κόμβου
print(network.buses_t.marginal_price)

Index(['Bus_A', 'Bus_B', 'Bus_C'], dtype='str', name='name')
Index(['Line_AB', 'Line_BC'], dtype='str', name='name')
Index(['Line_AB', 'Line_BC'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 11 duals
Objective: 3.50e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.



--- ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ (MW) ---
name        g_A  Component_C
snapshot                    
0         500.0        100.0

--- ΡΟΕΣ ΙΣΧΥΟΣ ΣΤΙΣ ΓΡΑΜΜΕΣ (MW) ---
name      Line_AB  Line_BC
snapshot                  
0           300.0    300.0

--- ΟΡΙΑΚΕΣ ΤΙΜΕΣ ΚΟΜΒΩΝ (LMP σε €/MWh) ---
name      Bus_A  Bus_B  Bus_C
snapshot                     
0          50.0  100.0  100.0


Η Γραμμή Α-Β θα "τερματίσει": Στις ροές ισχύος θα δεις ότι η Line_AB μεταφέρει ακριβώς 300 MW. Το καλώδιο γέμισε 100%.

    Η Φθηνή Γεννήτρια (g_A) θα μειώσει παραγωγή: Αντί για 600 MW που παρήγαγε πριν, τώρα θα παράγει μόνο 500 MW (200 MW για την πόλη της + 300 MW που χωράνε να φύγουν από το καλώδιο).

    Η Ακριβή Γεννήτρια (Component_C) μπαίνει στο παιχνίδι: Αφού η πόλη C χρειάζεται 400 MW, αλλά από το καλώδιο έρχονται μόνο 300 MW, τα υπόλοιπα 100 MW πρέπει να παραχθούν τοπικά. Άρα η ακριβή γεννήτρια των 100 €/MWh θα πάρει μπροστά!

Αυτός είναι ο λόγος που σε πραγματικά δίκτυα (όπως τα νησιά που δεν είναι καλά διασυνδεδεμένα με την ηπειρωτική χώρα), το ρεύμα κοστίζει πολύ περισσότερο για να παραχθεί: επειδή τα καλώδια δεν επαρκούν για να φέρουν τη φθηνή ενέργεια και αναγκάζονται να καίνε ακριβό πετρέλαιο τοπικά.